In [3]:
import sys, os

import numpy as np
import matplotlib.pyplot as plt

from env.drone_env        import DroneEnv2D
from env.discrete_wrapper import DiscreteWrapper


In [4]:
def make_env(level=0, seed=42):
    base = DroneEnv2D(world_size=20.0, max_steps=300,
                      curriculum_level=level, seed=seed)
    return DiscreteWrapper(base, grid_size=20)


env = make_env(level=0)
print(f'States: {env.n_states}  Actions: {env.n_actions}')



States: 400  Actions: 9


## 1. Sanity check — random policy

> Establishes a baseline. Every algorithm below should beat this.


In [ ]:
rewards = []
for _ in range(200):
    s, _ = env.reset()
    G = 0.0
    while True:
        a = env.action_space.sample()
        s, r, t1, t2, _ = env.step(a)
        G += r
        if t1 or t2:
            break
    rewards.append(G)

print(f'Random policy  mean reward: {np.mean(rewards):.2f}')


Random policy  mean reward: -8.78


## 2. Dynamic Programming — Policy Iteration & Value Iteration

Requires a model. We approximate P(s'|s,a) and R(s,a) by sampling
the discrete env exhaustively, then solve the Bellman equations exactly.

Note: this builds a model over 400 states × 9 actions, which takes
a couple of minutes. Skip this cell if you're short on time —
the other algorithms don't depend on it.

In [ ]:

# %%
from algos.classical.dynamic_programming import build_model, policy_iteration, value_iteration
from algos.classical.dynamic_programming import policy_improvement

dp_env = make_env(level=0)
P, R   = build_model(dp_env, n_samples=3)

pi_policy, V_pi = policy_iteration(P, R, gamma=0.99, theta=1e-4)
vi_policy, V_vi = value_iteration(P, R, gamma=0.99, theta=1e-4)

print(f'\nV_pi range : [{V_pi.min():.2f}, {V_pi.max():.2f}]')
print(f'V_vi range : [{V_vi.min():.2f}, {V_vi.max():.2f}]')


Building environment model...


States: 100%|██████████| 400/400 [00:08<00:00, 47.94it/s]



=== Policy Iteration ===
  Iteration 1
  Policy evaluation converged in 1158 sweeps
  Iteration 2
  Policy evaluation converged in 1267 sweeps
  Iteration 3
  Policy evaluation converged in 1267 sweeps
  Iteration 4
  Policy evaluation converged in 1267 sweeps
  Iteration 5
  Policy evaluation converged in 1267 sweeps
  Iteration 6
  Policy evaluation converged in 1267 sweeps
  Iteration 7
  Policy evaluation converged in 1267 sweeps
  Iteration 8
  Policy evaluation converged in 1267 sweeps
  Iteration 9
  Policy evaluation converged in 1267 sweeps
  Iteration 10
  Policy evaluation converged in 1267 sweeps
  Iteration 11
  Policy evaluation converged in 1267 sweeps
  Iteration 12
  Policy evaluation converged in 1267 sweeps
  Iteration 13
  Policy evaluation converged in 1267 sweeps
  Iteration 14
  Policy evaluation converged in 1267 sweeps
  Iteration 15
  Policy evaluation converged in 1267 sweeps
  Iteration 16
  Policy evaluation converged in 1267 sweeps
  Iteration 17
  Policy

In [ ]:

# # %% [markdown]
# # ## 1. Sanity check — random policy

# # Establishes a baseline. Every algorithm below should beat this.

# # %%
# rewards = []
# for _ in range(200):
#     s, _ = env.reset()
#     G = 0.0
#     while True:
#         a = env.action_space.sample()
#         s, r, t1, t2, _ = env.step(a)
#         G += r
#         if t1 or t2:
#             break
#     rewards.append(G)

# print(f'Random policy  mean reward: {np.mean(rewards):.2f}')


# # %% [markdown]
# # ## 2. Q-Learning
# #
# # Off-policy TD control.
# # δ = r + γ max_a' Q(s',a') - Q(s,a)

# # %%
# from algos.classical import QLearning

# ql = QLearning(env.n_states, env.n_actions, gamma=0.99, alpha=0.1)
# ql.train(make_env(), n_episodes=3000, log_every=300)


# # %% [markdown]
# # ## 3. SARSA — compare with Q-Learning
# #
# # On-policy TD control.
# # δ = r + γ Q(s',a') - Q(s,a)   where a' is sampled from ε-greedy

# # %%
# from algos.classical import SARSA

# sarsa = SARSA(env.n_states, env.n_actions, gamma=0.99, alpha=0.1)
# sarsa.train(make_env(), n_episodes=3000, log_every=300)


# # %% [markdown]
# # ## 4. Monte Carlo
# #
# # Model-free, learns from complete episodes.
# # Q(s,a) ← Q(s,a) + α [G_t - Q(s,a)]

# # %%
# from algos.classical import MonteCarloControl

# mc = MonteCarloControl(env.n_states, env.n_actions,
#                        gamma=0.99, alpha=0.05, first_visit=True)
# mc.train(make_env(), n_episodes=2000, log_every=200)


# # %% [markdown]
# # ## 5. Dynamic Programming — Policy Iteration & Value Iteration
# #
# # Requires a model. We approximate P(s'|s,a) and R(s,a) by sampling
# # the discrete env exhaustively, then solve the Bellman equations exactly.
# #
# # Note: this builds a model over 400 states × 9 actions, which takes
# # a couple of minutes. Skip this cell if you're short on time —
# # the other algorithms don't depend on it.

# # %%
# from algos.classical import build_model, policy_iteration, value_iteration
# from algos.classical.dynamic_programming import policy_improvement

# dp_env = make_env(level=0)
# P, R   = build_model(dp_env, n_samples=3)

# pi_policy, V_pi = policy_iteration(P, R, gamma=0.99, theta=1e-4)
# vi_policy, V_vi = value_iteration(P, R, gamma=0.99, theta=1e-4)

# print(f'\nV_pi range : [{V_pi.min():.2f}, {V_pi.max():.2f}]')
# print(f'V_vi range : [{V_vi.min():.2f}, {V_vi.max():.2f}]')


# # %% [markdown]
# # ## 6. Learning curves — all online algorithms
# #
# # Things to look for:
# # - **Q-Learning** often rises fastest early — more aggressive off-policy learning.
# # - **SARSA** may be slightly lower but more stable — it accounts for its own exploration.
# # - **Monte Carlo** starts slower — needs full episodes before any update happens.

# # %%
# def smooth(x, w=50):
#     return np.convolve(x, np.ones(w) / w, mode='valid')

# fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# for agent, label in [(ql, 'Q-Learning'), (sarsa, 'SARSA'), (mc, 'Monte Carlo')]:
#     ax1.plot(smooth(agent.episode_rewards), label=label)
#     ax2.plot(smooth(agent.episode_lengths), label=label)

# for ax, title, ylabel in [(ax1, 'Episode Reward', 'reward'),
#                            (ax2, 'Episode Length', 'steps')]:
#     ax.set_title(title)
#     ax.set_xlabel('episode')
#     ax.set_ylabel(ylabel)
#     ax.legend()
#     ax.grid(alpha=0.3)

# plt.tight_layout()
# plt.show()


# # %% [markdown]
# # ## 7. Visualise the Q-table as a heatmap + policy arrows
# #
# # V(s) = max_a Q(s,a)  — shows how "good" each grid cell is.
# # π*(s) = argmax_a Q(s,a)  — shows which direction the agent thrusts from each cell.

# # %%
# V  = np.max(ql.Q, axis=1).reshape(env.grid_size, env.grid_size)
# PI = np.argmax(ql.Q, axis=1).reshape(env.grid_size, env.grid_size)

# # Action index -> (dx, dy) direction, matching DiscreteWrapper.ACTION_VECTORS
# DX = [ 0, 1, 1, 1, 0, -1, -1, -1, 0]
# DY = [-1, -1, 0, 1, 1,  1,  0, -1, 0]

# fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# # Value heatmap
# im = axes[0].imshow(V, origin='upper', cmap='viridis')
# plt.colorbar(im, ax=axes[0])
# axes[0].set_title('V(s) = max_a Q(s,a)  —  Q-Learning')

# # Policy arrows over a faded value map
# axes[1].imshow(V, origin='upper', cmap='viridis', alpha=0.4)
# g = env.grid_size
# for row in range(g):
#     for col in range(g):
#         a = PI[row, col]
#         if a < 8:   # skip hover (action 8 has zero direction)
#             axes[1].annotate(
#                 '', xy=(col + 0.5 * DX[a], row + 0.5 * DY[a]),
#                 xytext=(col, row),
#                 arrowprops=dict(arrowstyle='->', color='white', lw=0.8),
#             )
# axes[1].set_title('Greedy policy  π*(s) = argmax_a Q(s,a)')

# plt.tight_layout()
# plt.show()


# # %% [markdown]
# # ## 8. Key observations
# #
# # - The **V(s) heatmap** should show high values near where the goal tends
# #   to spawn, and low values near obstacles/borders.
# # - The **policy arrows** should generally point toward higher-value regions.
# # - All three online algorithms should comfortably beat the random baseline
# #   from section 1.
# #
# # The plateau you'll observe here is the ceiling of tabular RL: velocity,
# # obstacle rays, and fine-grained position are all invisible to the agent
# # (we only kept a 20x20 grid over x,y). That's the motivation for Phase 2 —
# # deep RL, where a neural network can consume the full 14-dim continuous state.